# TCS iON Voice AI -- Pipeline Walkthrough

Runs the full Appendix A pipeline (ASR/gesture -> merge -> language/accent/disfluency analysis -> normalize -> accessibility report -> intent -> RAG -> grounded response -> recovery decision) on one of the PRD's own demo samples.

Uses `mock` backends so this runs anywhere with no API key, no model download, and no mic/camera hardware -- see `../README.md` for switching to real backends (`local`/`ollama`/`event`).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.pipeline import run_pipeline

# PRD demo sample 1: "B-b-b-book appointment tomorrow"
result = run_pipeline(text_override='B-b-b-book appointment tomorrow')
print('Pipeline ran. Fallback used:', result.used_fallback_cache)

Pipeline ran. Fallback used: False


## Original input vs. accessible transcript (FR-04, FR-07)

In [2]:
print('Original:', result.original_input.text)
print('Confidence:', result.original_input.confidence)
print('Accessible transcript:', result.accessible_transcript.text)
print('Corrections made:', result.accessible_transcript.corrections_made)

Original: B-b-b-book appointment tomorrow
Confidence: 1.0
Accessible transcript: B-book appointment tomorrow
Corrections made: True


## Disfluency / language / accent analysis (FR-03, FR-05, FR-06)

In [3]:
print('Disfluency report:', result.disfluency_report)
print('Language report:', result.language_report)
print('Accent/noise report:', result.accent_noise_report)

Disfluency report: DisfluencyReport(has_disfluency=True, repeated_syllables=['B-b-b-book'], repeated_words=['b'], filler_words_found=[], long_pause_markers=0)
Language report: LanguageReport(languages_detected=['en'], code_mixed=False, romanized_markers=[])
Accent/noise report: AccentNoiseReport(confidence=1.0, is_low_confidence=False, clarifying_question=None)


## Structured accessibility report (FR-09)

In [4]:
print('Barriers detected:', result.accessibility_report.barriers_detected)
print('Support applied:', result.accessibility_report.support_applied)
print('Notes:', result.accessibility_report.notes)

Barriers detected: ['speech_impairment']
Support applied: ['speech impairment support']
Notes: supportive assistance, not a diagnosis


## Visual equivalence (FR-17) and intent (FR-11)

In [5]:
print('Caption:', result.visual_equivalent.caption)
print('Action preview:', result.visual_equivalent.action_preview)
print('Intent goal:', result.intent.goal)
print('Intent action type:', result.intent.action_type)
print('Missing information:', result.intent.missing_information)

Caption: B-book appointment tomorrow
Action preview: About to act on: B-book appointment tomorrow
Intent goal: unknown
Intent action type: unknown
Missing information: []


## RAG-retrieved context (FR-12) and final grounded response (FR-13)

In [6]:
for snippet in result.retrieved_context:
    print('-', snippet.title, '(', snippet.source, ')')
print()
print('Final response:', result.final_response.text)

- Principle 2 - Simplify ( PAS 901:2025 )
- Is my audio or video stored? ( Accessibility FAQ )
- Principle 1 - Understand ( PAS 901:2025 )

Final response: Understood: B-book appointment tomorrow


## Agent reasoning trace (the agentic flow)

The agent plans, calls skills as tools, observes each result, and decides to call another, ask a clarifying question, or finish — a real multi-step loop, not one-shot routing. On a FAQ-style question it calls the `faq_lookup` tool, then finalizes.

In [7]:
faq_result = run_pipeline(text_override='What is PAS 901?')
agent = faq_result.agent_result
print('Steps:', len(agent.steps), '| tools used:', agent.skills_used)
for s in agent.steps:
    print(f'  Step {s.step}: {s.action}' + (f' -> {s.skill}' if s.skill else ''))
    print(f'    thought: {s.thought}')
    if s.observation:
        print(f'    observation: {s.observation}')
print('Agent answer:', agent.answer)

Steps: 2 | tools used: ['faq_lookup']
  Step 1: call_skill -> faq_lookup
    thought: 'What is PAS 901?' matches the faq_lookup tool; calling it.
    observation: Build trust through transparency and privacy. Privacy should be integrated by design, systems secure by default, and users should always know what data is collected, why, and how it's used.
  Step 2: finish
    thought: I have the tool result; finalizing.
Agent answer: Based on that: Build trust through transparency and privacy. Privacy should be integrated by design, systems secure by default, and users should always know what data is collected, why, and how it's used.


## Recovery / confirmation decision (FR-16)

In [8]:
print('Needs confirmation:', result.recovery_options.needs_confirmation)
print('Options:', result.recovery_options.options)
print('Reason:', result.recovery_options.reason)

Needs confirmation: False
Options: ['confirm']
Reason: confidence acceptable


## PRD demo sample 3: accent uncertainty ("eleven" -> "Did you mean floor 11?")

In [9]:
from src.input.merge import merge_inputs
from src.input.microphone import Transcript
from src.analysis.accent_noise import analyze_accent_noise_confidence
from src.llm import get_llm_backend

ctx = merge_inputs(transcript=Transcript(text='eleven', confidence=0.3))
report = analyze_accent_noise_confidence(ctx, llm=get_llm_backend('reasoning'))
print('Clarifying question:', report.clarifying_question)

Clarifying question: Did you mean floor 11?


## PRD demo sample 8: predefined gesture interaction (FR-14)

In [10]:
from src.input.vision import MockVisionBackend

vision = MockVisionBackend()
gesture = vision.interpret('../data/sample_gestures/thumbs_up.json')
print('Gesture:', gesture.gesture, '-> candidate intent:', gesture.candidate_intent)

Gesture: thumbs_up -> candidate intent: confirm


## Resilience proof (NFR-03): cached scenario fallback if live backends fail

See `tests/test_pipeline_smoke.py::test_cached_fallback_scenario_used_when_pipeline_fails` for an automated test that simulates a total backend outage and confirms the pipeline falls back to a precomputed `data/demo_scenarios.json` entry instead of crashing.

## Extensibility proof (FR-20)

See `tests/test_pipeline_smoke.py::test_new_skill_can_be_added_without_touching_orchestrator` -- adds a brand-new domain skill file at runtime and confirms it's auto-registered with zero changes to the pipeline.